<a href="https://colab.research.google.com/github/Gayathri-rfr/DataPOC/blob/main/data_analystics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import database

def run_pipeline():

    # 1. Pull data directly from Django simulation layer
    raw_records = database.get_django_records()
    if not raw_records:
        print("Data pipeline empty. Please run 'run_test.py' or hit the API first.")
        return

    df = pd.DataFrame(raw_records)
    print(f"🔹 Successfully extracted {len(df)} records from database.")

    # 2. Pandas feature engineering & NumPy transformations
    df['amount'] = df['amount'].fillna(0.0)
    df['log_amount'] = np.log1p(df['amount'])  # Apply NumPy matrix transformations

    print("\n--- Processed Feature Matrix Sample ---")
    print(df[['tx_id', 'amount', 'log_amount', 'is_anomaly']].head())

    # 3. Batch Aggregations (Replacing the Spark layer)
    historical_summary = df.groupby("category").agg(
        total_revenue=('amount', 'sum'),
        average_tx_value=('amount', 'mean'),
        detected_anomalies=('is_anomaly', 'sum')
    ).reset_index()

    print("\n--- Aggregated Analytical Summary Report ---")
    print(historical_summary.to_string(index=False))

    # Save physical asset
    output_filename = "category_analytics_report.csv"
    historical_summary.to_csv(output_filename, index=False)
    print(f"\n Analytical report compiled and written to: '{output_filename}'\n")

if __name__ == "__main__":
    run_pipeline()